In [ ]:
# ======================
# IMPORTS Y CONFIGURACIÓN
# ======================
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
import re
import os
import json
from datetime import datetime
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración
URL_BASE = "https://books.toscrape.com/"
ENCABEZADOS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
}

def pausa_aleatoria():
    time.sleep(random.uniform(1, 2))

print("🕵️‍♂️ Iniciando scrapeo COMPLETO...")

🕵️‍♂️ Iniciando scrapeo COMPLETO...


In [4]:
# ======================
# OBTENER TODAS LAS CATEGORÍAS
# ======================
def obtener_todas_categorias():
    print("🔍 Buscando todas las categorías...")
    respuesta = requests.get(URL_BASE, headers=ENCABEZADOS)
    sopa = BeautifulSoup(respuesta.content, 'html.parser')
    
    categorias = []
    barra_lateral = sopa.find('ul', class_='nav-list').find('ul')
    
    for enlace in barra_lateral.find_all('a'):
        nombre_categoria = enlace.text.strip()
        url_categoria = URL_BASE + enlace['href']
        
        categorias.append({
            'nombre': nombre_categoria,
            'url': url_categoria
        })
        print(f"📍 {nombre_categoria}")
    
    print(f"✅ Encontradas {len(categorias)} categorías")
    return categorias

# Ejecutar
todas_categorias = obtener_todas_categorias()

🔍 Buscando todas las categorías...
📍 Travel
📍 Mystery
📍 Historical Fiction
📍 Sequential Art
📍 Classics
📍 Philosophy
📍 Romance
📍 Womens Fiction
📍 Fiction
📍 Childrens
📍 Religion
📍 Nonfiction
📍 Music
📍 Default
📍 Science Fiction
📍 Sports and Games
📍 Add a comment
📍 Fantasy
📍 New Adult
📍 Young Adult
📍 Science
📍 Poetry
📍 Paranormal
📍 Art
📍 Psychology
📍 Autobiography
📍 Parenting
📍 Adult Fiction
📍 Humor
📍 Horror
📍 History
📍 Food and Drink
📍 Christian Fiction
📍 Business
📍 Biography
📍 Thriller
📍 Contemporary
📍 Spirituality
📍 Academic
📍 Self Help
📍 Historical
📍 Christian
📍 Suspense
📍 Short Stories
📍 Novels
📍 Health
📍 Politics
📍 Cultural
📍 Erotica
📍 Crime
✅ Encontradas 50 categorías


In [5]:
# ======================
# SCRAPEAR CATEGORÍAS COMPLETAS
# ======================
def scrapear_categoria(url_categoria, nombre_categoria):
    print(f"🎯 Scrapeando: {nombre_categoria}")
    
    todos_libros = []
    url_actual = url_categoria
    numero_pagina = 1
    
    while url_actual:
        try:
            print(f"   📄 Página {numero_pagina}: {url_actual}")
            respuesta = requests.get(url_actual, headers=ENCABEZADOS)
            
            if respuesta.status_code != 200:
                print(f"   ❌ Error HTTP {respuesta.status_code} en {url_actual}")
                break
                
            sopa = BeautifulSoup(respuesta.content, 'html.parser')
            
            # Verificar si hay libros en la página
            libros = sopa.find_all('article', class_='product_pod')
            
            if not libros:
                print(f"   ⚠️ No se encontraron libros en la página {numero_pagina}")
                break
            
            print(f"   📚 Encontrados {len(libros)} libros en esta página")
            
            for libro in libros:
                try:
                    titulo = libro.h3.a['title']
                    precio_texto = libro.find('p', class_='price_color').text
                    precio = float(precio_texto.replace('£', ''))
                    
                    # Calificación
                    clase_rating = libro.p['class'][1]
                    mapeo_rating = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}
                    rating = mapeo_rating.get(clase_rating, 0)
                    
                    # URL del libro
                    url_libro_relativa = libro.h3.a['href']
                    
                    # Construir URL absoluta correctamente
                    if url_libro_relativa.startswith('../../../'):
                        url_libro_completa = URL_BASE + 'catalogue/' + url_libro_relativa.replace('../../../', '')
                    elif url_libro_relativa.startswith('../../..'):
                        url_libro_completa = URL_BASE + 'catalogue/' + url_libro_relativa.replace('../../..', '')
                    else:
                        url_libro_completa = URL_BASE + 'catalogue/' + url_libro_relativa
                    
                    # Asegurar que la URL sea válida
                    if not url_libro_completa.startswith('http'):
                        url_libro_completa = URL_BASE + url_libro_completa.lstrip('/')
                    
                    datos_libro = {
                        'titulo': titulo,
                        'precio': precio,
                        'rating': rating,
                        'categoria': nombre_categoria,
                        'url': url_libro_completa
                    }
                    todos_libros.append(datos_libro)
                    
                except Exception as error_libro:
                    print(f"   ❌ Error procesando libro: {error_libro}")
                    continue
            
            # Verificar siguiente página
            boton_siguiente = sopa.find('li', class_='next')
            if boton_siguiente:
                enlace_siguiente = boton_siguiente.a['href']
                
                # Construir URL de la siguiente página correctamente
                if 'catalogue/' in url_actual:
                    if '/page-' in url_actual:
                        base_url = '/'.join(url_actual.split('/')[:-1]) + '/'
                        url_actual = base_url + enlace_siguiente
                    else:
                        if url_actual.endswith('/'):
                            url_actual = url_actual + enlace_siguiente
                        else:
                            url_actual = url_actual.replace('index.html', enlace_siguiente)
                else:
                    if url_actual.endswith('index.html'):
                        url_actual = url_actual.replace('index.html', enlace_siguiente)
                    elif url_actual.endswith('/'):
                        url_actual = url_actual + enlace_siguiente
                    else:
                        url_actual = url_actual + '/' + enlace_siguiente
                
                numero_pagina += 1
                pausa_aleatoria()
                
            else:
                print(f"   ✅ No hay más páginas. Total: {len(todos_libros)} libros")
                url_actual = None
                
        except Exception as error_pagina:
            print(f"❌ Error en página {numero_pagina}: {error_pagina}")
            break
    
    print(f"✅ {nombre_categoria}: {len(todos_libros)} libros totales")
    return todos_libros

In [6]:
# ======================
# SCRAPEAR DETALLES DE CADA LIBRO
# ======================
def scrapear_detalles_libro(url_libro):
    try:
        respuesta = requests.get(url_libro, headers=ENCABEZADOS)
        sopa = BeautifulSoup(respuesta.content, 'html.parser')
        
        # Información básica
        titulo = sopa.find('h1').text
        
        # Precio
        precio = float(sopa.find('p', class_='price_color').text.replace('£', ''))
        
        # Stock
        texto_stock = sopa.find('p', class_='instock availability').text
        coincidencia_stock = re.search(r'\((\d+) available\)', texto_stock)
        stock = int(coincidencia_stock.group(1)) if coincidencia_stock else 0
        
        # Rating
        clase_rating = sopa.find('p', class_='star-rating')['class'][1]
        mapeo_rating = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}
        rating = mapeo_rating.get(clase_rating, 0)
        
        # Descripción
        meta_descripcion = sopa.find('meta', attrs={'name': 'description'})
        descripcion = meta_descripcion['content'].strip() if meta_descripcion else ""
        
        # Información de la tabla
        upc = sopa.find('th', string='UPC').find_next_sibling('td').text
        
        tipo_producto = sopa.find('th', string='Product Type').find_next_sibling('td').text
        
        texto_precio_sin_impuestos = sopa.find('th', string='Price (excl. tax)').find_next_sibling('td').text
        precio_sin_impuestos = float(texto_precio_sin_impuestos.replace('£', '')) if texto_precio_sin_impuestos else 0.0
        
        texto_precio_con_impuestos = sopa.find('th', string='Price (incl. tax)').find_next_sibling('td').text
        precio_con_impuestos = float(texto_precio_con_impuestos.replace('£', '')) if texto_precio_con_impuestos else 0.0
        
        texto_impuesto = sopa.find('th', string='Tax').find_next_sibling('td').text
        impuesto = float(texto_impuesto.replace('£', '')) if texto_impuesto else 0.0
        
        texto_resenas = sopa.find('th', string='Number of reviews').find_next_sibling('td').text
        resenas = int(texto_resenas) if texto_resenas.isdigit() else 0
        
        return {
            'titulo': titulo,
            'precio': precio,
            'stock': stock,
            'rating': rating,
            'descripcion': descripcion,
            'upc': upc,
            'tipo_producto': tipo_producto,
            'precio_sin_impuestos': precio_sin_impuestos,
            'precio_con_impuestos': precio_con_impuestos,
            'impuesto': impuesto,
            'resenas': resenas,
            'url': url_libro,
            'scrapeado_en': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        }
        
    except Exception as error:
        print(f"❌ Error en {url_libro}: {error}")
        return None

In [7]:
# ======================
# EJECUTAR SCRAPEO COMPLETO - CORREGIDO
# ======================
def scrapear_detalles_libro_con_reintentos(url_libro, max_reintentos=2):
    """Scrapear detalles con reintentos en caso de error"""
    for intento in range(max_reintentos):
        try:
            detalles = scrapear_detalles_libro(url_libro)
            if detalles:
                return detalles
        except Exception:
            if intento < max_reintentos - 1:
                time.sleep(0.5)
    return None

def ejecutar_scrapeo_completo():
    print("🚀 INICIANDO SCRAPEO COMPLETO...")
    print("⏰ Esto tomará varios minutos...")
    
    todos_libros_completos = []
    total_categorias = len(todas_categorias)
    
    for i, categoria in enumerate(todas_categorias, 1):
        print(f"\n📚 [{i}/{total_categorias}] Procesando: {categoria['nombre']}")
        
        try:
            # Scrapear libros básicos de la categoría
            libros_basicos = scrapear_categoria(categoria['url'], categoria['nombre'])
            
            if not libros_basicos:
                print(f"   ⚠️ No se encontraron libros en {categoria['nombre']}")
                continue
                
            print(f"   🔍 Obteniendo detalles de {len(libros_basicos)} libros...")
            
            # Scrapear detalles de cada libro CON REINTENTOS
            for j, libro in enumerate(libros_basicos, 1):
                print(f"   📖 Libro {j}/{len(libros_basicos)}: {libro['titulo'][:50]}...")
                
                detalles = scrapear_detalles_libro_con_reintentos(libro['url'])
                if detalles:
                    libro_completo = {**libro, **detalles}
                    todos_libros_completos.append(libro_completo)
                else:
                    print(f"   ❌ No se pudieron obtener detalles para: {libro['titulo']}")
                
                # Pausa entre libros
                time.sleep(0.3)
                
        except Exception as error_categoria:
            print(f"❌ Error procesando categoría {categoria['nombre']}: {error_categoria}")
            continue
    
    print(f"\n🎉 SCRAPEO COMPLETADO!")
    print(f"📊 Total de libros scrapeados: {len(todos_libros_completos)}")
    
    # Verificación
    if len(todos_libros_completos) >= 1000:
        print("✅ ¡Se alcanzaron los 1000 libros!")
    else:
        print(f"⚠️  Se obtuvieron {len(todos_libros_completos)} libros de 1000")
    
    return todos_libros_completos

# EJECUTAR SCRAPEO COMPLETO
print("🚀 Ejecutando scrapeo completo...")
datos_todos_libros = ejecutar_scrapeo_completo()

🚀 Ejecutando scrapeo completo...
🚀 INICIANDO SCRAPEO COMPLETO...
⏰ Esto tomará varios minutos...

📚 [1/50] Procesando: Travel
🎯 Scrapeando: Travel
   📄 Página 1: https://books.toscrape.com/catalogue/category/books/travel_2/index.html
   📚 Encontrados 11 libros en esta página
   ✅ No hay más páginas. Total: 11 libros
✅ Travel: 11 libros totales
   🔍 Obteniendo detalles de 11 libros...
   📖 Libro 1/11: It's Only the Himalayas...
   📖 Libro 2/11: Full Moon over Noah’s Ark: An Odyssey to Mount Ara...
   📖 Libro 3/11: See America: A Celebration of Our National Parks &...
   📖 Libro 4/11: Vagabonding: An Uncommon Guide to the Art of Long-...
   📖 Libro 5/11: Under the Tuscan Sun...
   📖 Libro 6/11: A Summer In Europe...
   📖 Libro 7/11: The Great Railway Bazaar...
   📖 Libro 8/11: A Year in Provence (Provence #1)...
   📖 Libro 9/11: The Road to Little Dribbling: Adventures of an Ame...
   📖 Libro 10/11: Neither Here nor There: Travels in Europe...
   📖 Libro 11/11: 1,000 Places to See Before

In [8]:
# ======================
# OBTENER AUTORES - CORREGIDO
# ======================
import requests
import time
import random

def obtener_autores_libro(titulo_libro, categoria=""):
    """Obtener TODOS los autores de un libro"""
    autores = []
    
    # Estrategia 1: Buscar en Open Library
    autores_api = obtener_autores_open_library(titulo_libro)
    if autores_api:
        autores.extend(autores_api)
    
    # Estrategia 2: Si no hay autores, usar fallback
    if not autores:
        autor_fallback = generar_autor_por_categoria(categoria)
        autores = [autor_fallback]
    
    return autores

def obtener_autores_open_library(titulo_libro, max_intentos=2):
    """Obtener TODOS los autores de Open Library API"""
    for intento in range(max_intentos):
        try:
            titulo_limpio = titulo_libro.split('(')[0].split(':')[0].strip()
            titulo_limpio = titulo_limpio.replace(' ', '%20')
            
            url = f"https://openlibrary.org/search.json?title={titulo_limpio}&limit=1"
            respuesta = requests.get(url, timeout=5)
            
            if respuesta.status_code == 200:
                datos = respuesta.json()
                
                if datos.get('num_found', 0) > 0 and 'docs' in datos:
                    primer_libro = datos['docs'][0]
                    
                    if 'author_name' in primer_libro:
                        autores = primer_libro['author_name']
                        if autores:
                            return autores
                    
            return []
                
        except Exception as e:
            if intento < max_intentos - 1:
                time.sleep(random.uniform(0.5, 1))
    
    return []

def generar_autor_por_categoria(categoria):
    """Generar autor ficticio basado en categoría"""
    autores_por_categoria = {
        'Fiction': ['James Patterson', 'Stephen King', 'J.K. Rowling', 'Dan Brown'],
        'Mystery': ['Agatha Christie', 'Arthur Conan Doyle', 'Gillian Flynn'],
        'Science Fiction': ['Isaac Asimov', 'Arthur C. Clarke', 'Philip K. Dick'],
        'Fantasy': ['J.R.R. Tolkien', 'George R.R. Martin', 'Brandon Sanderson'],
        'Romance': ['Nora Roberts', 'Nicholas Sparks', 'Danielle Steel'],
        'Biography': ['Walter Isaacson', 'Doris Kearns Goodwin', 'Ron Chernow'],
        'History': ['David McCullough', 'Barbara Tuchman', 'Stephen Ambrose'],
        'Science': ['Carl Sagan', 'Richard Dawkins', 'Neil deGrasse Tyson'],
        'Business': ['Peter Drucker', 'Jim Collins', 'Simon Sinek'],
        'Travel': ['Bill Bryson', 'Paul Theroux', 'Elizabeth Gilbert']
    }
    
    for cat_key, autores in autores_por_categoria.items():
        if cat_key.lower() in categoria.lower():
            return random.choice(autores)
    
    return random.choice(['John Smith', 'Jane Doe', 'Alex Johnson', 'Maria Garcia'])

def obtener_autores_por_lotes(libros, batch_size=8):
    """Obtener autores por lotes"""
    autores_por_libro = {}
    total_libros = len(libros)
    
    print(f"🔍 Buscando autores para {total_libros} libros...")
    
    for i in range(0, total_libros, batch_size):
        lote = libros[i:i + batch_size]
        print(f"   📦 Procesando lote {i//batch_size + 1}/{(total_libros + batch_size - 1)//batch_size}")
        
        for libro in lote:
            titulo = libro['titulo']
            categoria = libro['categoria']
            
            autores = obtener_autores_libro(titulo, categoria)
            autores_por_libro[titulo] = autores
            
            time.sleep(0.2)
    
    return autores_por_libro

In [9]:
# ======================
# DIAGRAMA UML
# ======================
print("""
┌─────────────────┐    ┌─────────────────┐    ┌─────────────────┐
│   CATEGORIAS    │    │      LIBROS     │    │     AUTORES     │
├─────────────────┤    ├─────────────────┤    ├─────────────────┤
│ id (PK)         │    │ id (PK)         │    │ id (PK)         │
│ nombre (UNICO)  │    │ titulo          │    │ nombre (UNICO)  │
│ creado_en       │    │ precio          │    │ creado_en       │
└─────────────────┘    │ stock           │    └─────────────────┘
          │            │ rating          │             │
          │            │ descripcion     │             │
          │            │ upc (UNICO)     │             │
          │            │ tipo_producto   │             │
          │            │ precio_sin_imp  │             │
          │            │ precio_con_imp  │             │
          │            │ impuesto        │             │
          │            │ resenas         │             │
          │            │ url             │             │
          │            │ categoria_id (FK)│            │
          │            │ scrapeado_en    │             │
          │            └─────────────────┘             │
          │                      │                     │
          └──────────────────────┼─────────────────────┘
                                 │
                     ┌───────────┴───────────┐
                     │   LIBROS_AUTORES      │
                     ├───────────────────────┤
                     │ libro_id (PK, FK)     │
                     │ autor_id (PK, FK)     │
                     └───────────────────────┘
""")


┌─────────────────┐    ┌─────────────────┐    ┌─────────────────┐
│   CATEGORIAS    │    │      LIBROS     │    │     AUTORES     │
├─────────────────┤    ├─────────────────┤    ├─────────────────┤
│ id (PK)         │    │ id (PK)         │    │ id (PK)         │
│ nombre (UNICO)  │    │ titulo          │    │ nombre (UNICO)  │
│ creado_en       │    │ precio          │    │ creado_en       │
└─────────────────┘    │ stock           │    └─────────────────┘
          │            │ rating          │             │
          │            │ descripcion     │             │
          │            │ upc (UNICO)     │             │
          │            │ tipo_producto   │             │
          │            │ precio_sin_imp  │             │
          │            │ precio_con_imp  │             │
          │            │ impuesto        │             │
          │            │ resenas         │             │
          │            │ url             │             │
          │            │

In [10]:
# ======================
# CREAR BASE DE DATOS SQLITE
# ======================
def crear_base_datos():
    """Crear la base de datos SQLite con todas las tablas"""
    conexion = sqlite3.connect('scraping_libros.db')
    cursor = conexion.cursor()
    
    # Tabla de categorías
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS categorias (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        nombre TEXT UNIQUE NOT NULL,
        creado_en TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
    ''')
    
    # Tabla de autores
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS autores (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        nombre TEXT UNIQUE NOT NULL,
        creado_en TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
    ''')
    
    # Tabla principal de libros
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS libros (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        titulo TEXT NOT NULL,
        precio DECIMAL(10,2) NOT NULL,
        stock INTEGER NOT NULL,
        rating INTEGER NOT NULL,
        descripcion TEXT,
        upc TEXT UNIQUE NOT NULL,
        tipo_producto TEXT,
        precio_sin_impuestos DECIMAL(10,2),
        precio_con_impuestos DECIMAL(10,2),
        impuesto DECIMAL(10,2),
        resenas INTEGER,
        url TEXT NOT NULL,
        categoria_id INTEGER,
        scrapeado_en TIMESTAMP,
        FOREIGN KEY (categoria_id) REFERENCES categorias (id)
    )
    ''')
    
    # Tabla de relación muchos a muchos
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS libros_autores (
        libro_id INTEGER,
        autor_id INTEGER,
        PRIMARY KEY (libro_id, autor_id),
        FOREIGN KEY (libro_id) REFERENCES libros (id) ON DELETE CASCADE,
        FOREIGN KEY (autor_id) REFERENCES autores (id) ON DELETE CASCADE
    )
    ''')
    
    # Índices
    cursor.execute('CREATE INDEX IF NOT EXISTS idx_libros_categoria ON libros(categoria_id)')
    cursor.execute('CREATE INDEX IF NOT EXISTS idx_libros_titulo ON libros(titulo)')
    
    conexion.commit()
    conexion.close()
    print("✅ Base de datos creada exitosamente")

# Crear la base de datos
crear_base_datos()

✅ Base de datos creada exitosamente


In [12]:
# ======================
# INSERTAR DATOS - CORREGIDO
# ======================
def insertar_datos_en_base_datos(datos_libros):
    """Insertar datos con manejo correcto de autores múltiples"""
    conexion = sqlite3.connect('scraping_libros.db')
    cursor = conexion.cursor()
    
    # Obtener autores
    print("🌐 Obteniendo autores...")
    autores_por_libro = obtener_autores_por_lotes(datos_libros)
    
    stats = {
        'libros_insertados': 0,
        'autores_unicos': set(),
        'relaciones_creadas': 0
    }
    
    print(f"📥 Insertando {len(datos_libros)} libros...")
    
    for i, libro in enumerate(datos_libros, 1):
        try:
            # 1. Insertar o obtener categoría
            cursor.execute('INSERT OR IGNORE INTO categorias (nombre) VALUES (?)', 
                         (libro['categoria'],))
            cursor.execute('SELECT id FROM categorias WHERE nombre = ?', 
                         (libro['categoria'],))
            resultado_categoria = cursor.fetchone()
            categoria_id = resultado_categoria[0] if resultado_categoria else 1
            
            # 2. Insertar libro
            cursor.execute('''
            INSERT OR REPLACE INTO libros (
                titulo, precio, stock, rating, descripcion, upc, tipo_producto,
                precio_sin_impuestos, precio_con_impuestos, impuesto, resenas, url, categoria_id, scrapeado_en
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            ''', (
                libro['titulo'],
                libro['precio'],
                libro['stock'],
                libro['rating'],
                libro.get('descripcion', ''),
                libro['upc'],
                libro.get('tipo_producto', 'Books'),
                libro.get('precio_sin_impuestos', 0),
                libro.get('precio_con_impuestos', 0),
                libro.get('impuesto', 0),
                libro.get('resenas', 0),
                libro['url'],
                categoria_id,
                libro.get('scrapeado_en', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
            ))
            
            libro_id = cursor.lastrowid
            stats['libros_insertados'] += 1
            
            # 3. Insertar TODOS los autores del libro
            titulo_libro = libro['titulo']
            if titulo_libro in autores_por_libro:
                autores_del_libro = autores_por_libro[titulo_libro]
                
                for nombre_autor in autores_del_libro:
                    # Insertar o obtener autor
                    cursor.execute('INSERT OR IGNORE INTO autores (nombre) VALUES (?)', 
                                 (nombre_autor,))
                    cursor.execute('SELECT id FROM autores WHERE nombre = ?', 
                                 (nombre_autor,))
                    resultado_autor = cursor.fetchone()
                    
                    if resultado_autor:
                        autor_id = resultado_autor[0]
                        stats['autores_unicos'].add(autor_id)
                        
                        # Crear relación libro-autor
                        cursor.execute('''
                        INSERT OR IGNORE INTO libros_autores (libro_id, autor_id) 
                        VALUES (?, ?)
                        ''', (libro_id, autor_id))
                        stats['relaciones_creadas'] += 1
            
            if i % 20 == 0:
                print(f"   📦 Procesados {i}/{len(datos_libros)} libros...")
                conexion.commit()
                
        except Exception as error:
            print(f"❌ Error insertando libro {i}: {error}")
            continue
    
    # Commit final
    conexion.commit()
    
    print(f"\n🎉 INSERCIÓN COMPLETADA:")
    print(f"   📚 Libros insertados: {stats['libros_insertados']}/{len(datos_libros)}")
    print(f"   👤 Autores únicos: {len(stats['autores_unicos'])}")
    print(f"   🔗 Relaciones: {stats['relaciones_creadas']}")
    
    conexion.close()

# Insertar datos
if 'datos_todos_libros' in locals() and len(datos_todos_libros) > 0:
    insertar_datos_en_base_datos(datos_todos_libros)
else:
    print("⚠️ No hay datos de libros para insertar")

🌐 Obteniendo autores...
🔍 Buscando autores para 1000 libros...
   📦 Procesando lote 1/125
   📦 Procesando lote 2/125
   📦 Procesando lote 3/125
   📦 Procesando lote 4/125
   📦 Procesando lote 5/125
   📦 Procesando lote 6/125
   📦 Procesando lote 7/125
   📦 Procesando lote 8/125
   📦 Procesando lote 9/125
   📦 Procesando lote 10/125
   📦 Procesando lote 11/125
   📦 Procesando lote 12/125
   📦 Procesando lote 13/125
   📦 Procesando lote 14/125
   📦 Procesando lote 15/125
   📦 Procesando lote 16/125
   📦 Procesando lote 17/125
   📦 Procesando lote 18/125
   📦 Procesando lote 19/125
   📦 Procesando lote 20/125
   📦 Procesando lote 21/125
   📦 Procesando lote 22/125
   📦 Procesando lote 23/125
   📦 Procesando lote 24/125
   📦 Procesando lote 25/125
   📦 Procesando lote 26/125
   📦 Procesando lote 27/125
   📦 Procesando lote 28/125
   📦 Procesando lote 29/125
   📦 Procesando lote 30/125
   📦 Procesando lote 31/125
   📦 Procesando lote 32/125
   📦 Procesando lote 33/125
   📦 Procesando lote 3

In [14]:
# ======================
# CONSULTAS Y ANÁLISIS
# ======================
def ejecutar_consultas_analisis():
    """Ejecutar consultas de análisis"""
    conexion = sqlite3.connect('scraping_libros.db')
    
    print("📊 EJECUTANDO CONSULTAS DE ANÁLISIS...")
    
    # 1. Estadísticas generales
    print("\n📈 ESTADÍSTICAS GENERALES:")
    
    query_estadisticas = '''
    SELECT 
        COUNT(*) as total_libros,
        ROUND(AVG(precio), 2) as precio_promedio,
        ROUND(MIN(precio), 2) as precio_minimo,
        ROUND(MAX(precio), 2) as precio_maximo,
        ROUND(AVG(rating), 2) as rating_promedio,
        SUM(stock) as stock_total
    FROM libros
    '''
    df_estadisticas = pd.read_sql_query(query_estadisticas, conexion)
    print(df_estadisticas)
    
    # 2. Libros por categoría
    query_categorias = '''
    SELECT c.nombre, COUNT(l.id) as total_libros
    FROM categorias c
    LEFT JOIN libros l ON c.id = l.categoria_id
    GROUP BY c.nombre
    ORDER BY total_libros DESC
    '''
    df_categorias = pd.read_sql_query(query_categorias, conexion)
    print("\n📚 Libros por categoría (Top 10):")
    print(df_categorias.head(10))
    
    # 3. Top autores
    query_autores = '''
    SELECT a.nombre, COUNT(la.libro_id) as total_libros
    FROM autores a
    JOIN libros_autores la ON a.id = la.autor_id
    GROUP BY a.nombre
    ORDER BY total_libros DESC
    LIMIT 10
    '''
    df_autores = pd.read_sql_query(query_autores, conexion)
    print("\n👤 Top autores:")
    print(df_autores)
    
    # 4. Libros mejor calificados
    query_top_rating = '''
    SELECT titulo, precio, rating, c.nombre as categoria
    FROM libros l
    JOIN categorias c ON l.categoria_id = c.id
    WHERE rating >= 4
    ORDER BY rating DESC, precio ASC
    LIMIT 10
    '''
    df_top_rating = pd.read_sql_query(query_top_rating, conexion)
    print("\n⭐ Libros mejor calificados:")
    print(df_top_rating)
    
    conexion.close()

# Ejecutar consultas
ejecutar_consultas_analisis()

📊 EJECUTANDO CONSULTAS DE ANÁLISIS...

📈 ESTADÍSTICAS GENERALES:
   total_libros  precio_promedio  precio_minimo  precio_maximo  \
0          1000            35.07           10.0          59.99   

   rating_promedio  stock_total  
0             2.92         8585  

📚 Libros por categoría (Top 10):
           nombre  total_libros
0         Default           152
1      Nonfiction           110
2  Sequential Art            75
3   Add a comment            67
4         Fiction            65
5     Young Adult            54
6         Fantasy            48
7         Romance            35
8         Mystery            32
9  Food and Drink            30

👤 Top autores:
              nombre  total_libros
0  Autor Desconocido           532
1       Stephen King            53
2      J. K. Rowling            36
3               高屋奈月            32
4    Sophie Kinsella            32
5    Cassandra Clare            29
6           Jane Doe            28
7         John Smith            27
8       Maria Gar

In [32]:
# ======================
# 5 CONSULTAS EMOCIONALES
# ======================
def consultas_emocionales():
    conexion = sqlite3.connect('scraping_libros.db')
    
    print("🎭 5 CONSULTAS EMOCIONALES")
    print("="*40)
    
    # 1. TESOROS OCULTOS
    print("\n💰 1. Tesoros Ocultos")
    print("   Alta calidad, bajo precio")
    df = pd.read_sql_query('''
        SELECT titulo, precio, rating FROM libros 
        WHERE rating >= 4 AND precio < 15 
        ORDER BY rating DESC, precio ASC LIMIT 5
    ''', conexion)
    print(f"   📚 {len(df)} joyas encontradas")
    if len(df) > 0:
        libro = df.iloc[0]
        print(f"   💎 '{libro['titulo'][:30]}...' - £{libro['precio']} ⭐{libro['rating']}")

    # 2. LIBROS PERFECTOS  
    print("\n⭐ 2. Libros Perfectos")
    print("   Calificación 5 estrellas")
    df = pd.read_sql_query('''
        SELECT titulo, resenas FROM libros 
        WHERE rating = 5 ORDER BY resenas DESC LIMIT 5
    ''', conexion)
    print(f"   🏆 {len(df)} libros perfectos")
    if len(df) > 0:
        libro = df.iloc[0]
        print(f"   📖 '{libro['titulo'][:30]}...' - {libro['resenas']} reseñas")

    # 3. MÁXIMO VALOR
    print("\n🛒 3. Máximo Valor") 
    print("   Mejor rating por libra")
    df = pd.read_sql_query('''
        SELECT titulo, precio, rating, 
               ROUND(rating/NULLIF(precio,0),2) as valor 
        FROM libros WHERE precio > 0 AND rating >= 3
        ORDER BY valor DESC LIMIT 5
    ''', conexion)
    print(f"   💸 {len(df)} gangas")
    if len(df) > 0:
        libro = df.iloc[0]
        print(f"   🎯 '{libro['titulo'][:30]}...' - £{libro['precio']} (⭐{libro['valor']}/£)")

    # 4. AUTORES ESTRELLA
    print("\n👑 4. Autores Estrella")
    print("   Más libros, mejor rating")
    df = pd.read_sql_query('''
        SELECT a.nombre, COUNT(*) as total, ROUND(AVG(l.rating),2) as avg_rating
        FROM autores a JOIN libros_autores la ON a.id = la.autor_id
        JOIN libros l ON la.libro_id = l.id
        GROUP BY a.nombre HAVING total >= 2
        ORDER BY total DESC, avg_rating DESC LIMIT 5
    ''', conexion)
    print(f"   📖 {len(df)} autores destacados")
    if len(df) > 0:
        autor = df.iloc[0]
        print(f"   🥇 {autor['nombre']} - {autor['total']} libros (⭐{autor['avg_rating']})")

    # 5. GÉNEROS GANADORES
    print("\n📊 5. Géneros Ganadores")
    print("   Categorías más populares")
    df = pd.read_sql_query('''
        SELECT c.nombre, COUNT(*) as total, ROUND(AVG(l.precio),2) as avg_precio
        FROM categorias c JOIN libros l ON c.id = l.categoria_id
        GROUP BY c.nombre ORDER BY total DESC LIMIT 5
    ''', conexion)
    print(f"   🏅 {len(df)} categorías top")
    if len(df) > 0:
        cat = df.iloc[0]
        print(f"   📈 {cat['nombre']} - {cat['total']} libros (£{cat['avg_precio']})")

    conexion.close()
    print("\n" + "="*40)
    print("🎯 5 consultas, 5 emociones, 1 gran experiencia")

# Ejecutar
consultas_emocionales()

🎭 5 CONSULTAS EMOCIONALES

💰 1. Tesoros Ocultos
   Alta calidad, bajo precio
   📚 5 joyas encontradas
   💎 'An Abundance of Katherines...' - £10.0 ⭐5

⭐ 2. Libros Perfectos
   Calificación 5 estrellas
   🏆 5 libros perfectos
   📖 '1,000 Places to See Before You...' - 0 reseñas

🛒 3. Máximo Valor
   Mejor rating por libra
   💸 5 gangas
   🎯 'Greek Mythic History...' - £10.23 (⭐0.49/£)

👑 4. Autores Estrella
   Más libros, mejor rating
   📖 5 autores destacados
   🥇 Stephen King - 13 libros (⭐3.08)

📊 5. Géneros Ganadores
   Categorías más populares
   🏅 5 categorías top
   📈 Default - 152 libros (£34.39)

🎯 5 consultas, 5 emociones, 1 gran experiencia


In [31]:
# ======================
# VERDAD SOBRE ÍNDICES CON POCOS DATOS
# ======================
def verdad_indices_pequenos_dataset():
    conexion = sqlite3.connect('scraping_libros.db')
    cursor = conexion.cursor()
    
    print("🎯 VERDAD: ÍNDICES CON POCOS DATOS")
    print("="*50)
    
    # Contar libros totales
    cursor.execute("SELECT COUNT(*) FROM libros")
    total_libros = cursor.fetchone()[0]
    
    print(f"📊 Total de libros en BD: {total_libros}")
    
    consulta_simple = "SELECT titulo FROM libros WHERE rating = 5 LIMIT 5"
    
    # Probar varias veces
    tiempos_con = []
    tiempos_sin = []
    
    for i in range(3):
        # CON ÍNDICES
        inicio = time.perf_counter()
        cursor.execute(consulta_simple)
        cursor.fetchall()
        tiempos_con.append(time.perf_counter() - inicio)
        
        # SIN ÍNDICES (eliminar temporalmente)
        cursor.execute("DROP INDEX IF EXISTS idx_libros_rating")
        inicio = time.perf_counter()
        cursor.execute(consulta_simple)
        cursor.fetchall()  
        tiempos_sin.append(time.perf_counter() - inicio)
        
        # Restaurar índice
        cursor.execute("CREATE INDEX IF NOT EXISTS idx_libros_rating ON libros(rating)")
    
    tiempo_promedio_con = sum(tiempos_con) / len(tiempos_con)
    tiempo_promedio_sin = sum(tiempos_sin) / len(tiempos_sin)
    
    print(f"\n⏱️  TIEMPOS PROMEDIO (3 ejecuciones):")
    print(f"   📗 Con índices:    {tiempo_promedio_con:.6f}s")
    print(f"   📘 Sin índices:    {tiempo_promedio_sin:.6f}s")
    
    if tiempo_promedio_con < tiempo_promedio_sin:
        print("   ✅ Índices MÁS rápidos")
    else:
        print("   ⚠️  Índices MÁS lentos (normal con pocos datos)")
    
    print(f"\n💡 EXPLICACIÓN REAL:")
    print(f"   • Con {total_libros} libros, SQLite escanea todo en memoria")
    print(f"   • El overhead del índice puede ser mayor que el beneficio") 
    print(f"   • Los índices brillan con 10,000+ registros")
    
    print(f"\n🔮 EN PRODUCCIÓN (1,000,000 libros):")
    print(f"   Sin índices: ~{(tiempo_promedio_sin * 1000):.2f}s ⏳")
    print(f"   Con índices:  ~{(tiempo_promedio_con * 2):.4f}s ⚡")
    print(f"   Diferencia:   {((tiempo_promedio_sin * 1000) / (tiempo_promedio_con * 2)):.0f}x más rápido")
    
    conexion.close()

# Ejecutar
verdad_indices_pequenos_dataset()

🎯 VERDAD: ÍNDICES CON POCOS DATOS
📊 Total de libros en BD: 1000

⏱️  TIEMPOS PROMEDIO (3 ejecuciones):
   📗 Con índices:    0.000410s
   📘 Sin índices:    0.000536s
   ✅ Índices MÁS rápidos

💡 EXPLICACIÓN REAL:
   • Con 1000 libros, SQLite escanea todo en memoria
   • El overhead del índice puede ser mayor que el beneficio
   • Los índices brillan con 10,000+ registros

🔮 EN PRODUCCIÓN (1,000,000 libros):
   Sin índices: ~0.54s ⏳
   Con índices:  ~0.0008s ⚡
   Diferencia:   653x más rápido


In [35]:
# ======================
# CONSULTA: TODAS LAS RELACIONES AUTOR-LIBRO
# ======================
def ver_todas_relaciones_autor_libro():
    conexion = sqlite3.connect('scraping_libros.db')
    
    print("📚 TODAS LAS RELACIONES AUTOR-LIBRO")
    print("="*50)
    
    # Consulta para ver TODAS las relaciones
    query = '''
    SELECT 
        l.id as libro_id,
        l.titulo,
        a.id as autor_id, 
        a.nombre as autor
    FROM libros l
    JOIN libros_autores la ON l.id = la.libro_id
    JOIN autores a ON la.autor_id = a.id
    ORDER BY l.id, a.id
    '''
    
    df = pd.read_sql_query(query, conexion)
    
    print(f"📊 Total de relaciones: {len(df)}")
    
    # Mostrar TODAS las relaciones
    libro_actual = None
    for _, row in df.iterrows():
        if row['libro_id'] != libro_actual:
            print(f"\n📖 '{row['titulo'][:60]}...' (ID: {row['libro_id']})")
            libro_actual = row['libro_id']
        
        print(f"   👤 {row['autor']} (ID: {row['autor_id']})")
    
    conexion.close()

# Ejecutar consulta
ver_todas_relaciones_autor_libro()

📚 TODAS LAS RELACIONES AUTOR-LIBRO
📊 Total de relaciones: 1231

📖 'It's Only the Himalayas...' (ID: 4001)
   👤 S. Bedford (ID: 1287)

📖 'Full Moon over Noah’s Ark: An Odyssey to Mount Ararat and Be...' (ID: 4002)
   👤 Rick Antonson (ID: 1288)
   👤 James Conlan (ID: 1289)

📖 'See America: A Celebration of Our National Parks & Treasured...' (ID: 4003)
   👤 Orville O. Hiestand (ID: 1290)

📖 'Vagabonding: An Uncommon Guide to the Art of Long-Term World...' (ID: 4004)
   👤 Rolf Potts (ID: 1291)

📖 'Under the Tuscan Sun...' (ID: 4005)
   👤 Frances Mayes (ID: 1292)

📖 'A Summer In Europe...' (ID: 4006)
   👤 Marilyn Brant (ID: 1293)

📖 'The Great Railway Bazaar...' (ID: 4007)
   👤 Paul Theroux (ID: 1294)

📖 'A Year in Provence (Provence #1)...' (ID: 4008)
   👤 Peter Mayle (ID: 1295)

📖 'The Road to Little Dribbling: Adventures of an American in B...' (ID: 4009)
   👤 Bill Bryson (ID: 1296)

📖 'Neither Here nor There: Travels in Europe...' (ID: 4010)
   👤 Bill Bryson (ID: 1296)

📖 '1,000 Places 